# Export m2-qwen3-4b → GGUF → HuggingFace (for Ollama)

Run this notebook on **Google Colab** (T4 GPU recommended, ~20GB disk needed).

At the end, anyone can run the model with:
```
ollama run hf.co/randyle1er/m2-qwen3-4b
```

## 1. Install dependencies

In [ ]:
!pip install -q tinker-ml tinker-cookbook huggingface_hub
!pip install -q transformers accelerate safetensors

# Clone llama.cpp for GGUF conversion + quantization
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

# Build the quantizer binary
!cmake llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize
print('Done.')

## 2. Set credentials

- `TINKER_API_KEY`: from your Tinker dashboard
- `HF_TOKEN`: HuggingFace token with **write** access — https://huggingface.co/settings/tokens

In [ ]:
import os

# Paste your keys here (or use Colab Secrets via the key icon in the sidebar)
os.environ['TINKER_API_KEY'] = 'tml-gnnuOReXJScfM3VqO62Rt9yVagpGPQQneb2ZQ9iYhw310e4QB4HUMf0ReHMrxlQTQAAAA'
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'  # <-- replace this
os.environ['HF_TRUST_REMOTE_CODE'] = '1'

TINKER_MODEL_PATH = 'tinker://23cdb1b1-10ca-5496-8afa-323e43af7aa6:train:0/sampler_weights/ephemeral_61'
BASE_MODEL        = 'Qwen/Qwen3.5-4B'
HF_REPO_ID        = 'randyle1er/m2-qwen3-4b'

ADAPTER_DIR = './adapter'
MERGED_DIR  = './merged_model'
GGUF_F16    = './model-f16.gguf'
GGUF_Q4     = './model-q4km.gguf'

## 3. Download adapter from Tinker

If this cell fails with a checkpoint-expired error, jump to the **Re-run training** section at the bottom first.

In [ ]:
from tinker_cookbook import weights

print('Downloading adapter from Tinker...')
adapter_dir = weights.download(
    tinker_path=TINKER_MODEL_PATH,
    output_dir=ADAPTER_DIR,
)
print(f'Adapter saved to: {adapter_dir}')
!ls -lh {adapter_dir}

## 4. Merge LoRA into base model

In [ ]:
print('Merging LoRA adapter into base model (downloads ~8GB, takes a few minutes)...')
weights.build_hf_model(
    base_model=BASE_MODEL,
    adapter_path=adapter_dir,
    output_path=MERGED_DIR,
    dtype='bfloat16',
    trust_remote_code=True,
)
print('Merged model saved.')
!du -sh {MERGED_DIR}

## 5. Convert to GGUF (f16)

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outtype f16 \
    --outfile {GGUF_F16}

!ls -lh {GGUF_F16}

## 6. Quantize to Q4_K_M (~2.5GB)

Q4_K_M is the standard Ollama quantization — good quality/size tradeoff.

In [ ]:
!./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M
!ls -lh {GGUF_Q4}

# Free up disk space now that we have the quantized file
!rm -f {GGUF_F16}
print('f16 GGUF deleted to free space.')

## 7. Push to HuggingFace

In [ ]:
from huggingface_hub import HfApi, ModelCard

api = HfApi(token=os.environ['HF_TOKEN'])

# Create the repo if it doesn't exist yet
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)

# Upload the GGUF
print(f'Uploading {GGUF_Q4} to {HF_REPO_ID}...')
api.upload_file(
    path_or_fileobj=GGUF_Q4,
    path_in_repo='model-q4km.gguf',
    repo_id=HF_REPO_ID,
    repo_type='model',
)

# Write a minimal model card so Ollama can detect the GGUF
card_content = f"""---
base_model: {BASE_MODEL}
language:
- en
license: apache-2.0
tags:
- macaulay2
- computational-algebra
- gguf
- ollama
---

# m2-qwen3-4b

Qwen3.5-4B fine-tuned on Macaulay2 input/output examples via LoRA SFT.
Given Macaulay2 code, the model predicts the exact output the interpreter would return.

## Usage

```bash
ollama run hf.co/{HF_REPO_ID}
```

## Training details

- Base model: `{BASE_MODEL}`
- Method: LoRA SFT (rank 16, lr 1e-4, 50 steps)
- Data: Macaulay2 examples from yulia-m2 dataset + Computations book
- Quantization: Q4_K_M
"""

card = ModelCard(card_content)
card.push_to_hub(HF_REPO_ID, token=os.environ['HF_TOKEN'])

print(f'\nDone! Model live at: https://huggingface.co/{HF_REPO_ID}')
print(f'\nRun with Ollama:')
print(f'  ollama run hf.co/{HF_REPO_ID}')

---
## Re-run training (if checkpoint expired)

If step 3 failed with a checkpoint-expired error, run this cell to re-fine-tune and grab a fresh model path.

In [ ]:
# Upload training data from your local machine first:
# In Colab: Files > Upload > select both JSON files from macaulay2-examples/
#   - m2_training_data.json
#   - computations_book_training_data.json

import asyncio, json, sys, numpy as np
import tinker
from tinker_cookbook.renderers import TrainOnWhat, get_renderer
from tinker_cookbook.supervised.data import conversation_to_datum
from pathlib import Path

DATA_PATHS = [
    Path('m2_training_data.json'),
    Path('computations_book_training_data.json'),
]

async def retrain():
    conversations = []
    for path in DATA_PATHS:
        if not path.exists():
            print(f'WARN: {path} not found, skipping')
            continue
        records = json.loads(path.read_text())
        conversations.extend(r['messages'] for r in records)
    print(f'Loaded {len(conversations)} examples')

    sc = tinker.ServiceClient(api_key=os.environ['TINKER_API_KEY'])
    training_client = await sc.create_lora_training_client_async(
        base_model='Qwen/Qwen3-4B', rank=16
    )
    tokenizer = training_client.get_tokenizer()
    renderer = get_renderer('qwen3_5', tokenizer)

    training_data = [
        conversation_to_datum(conv, renderer, max_length=512,
                              train_on_what=TrainOnWhat.LAST_ASSISTANT_MESSAGE)
        for conv in conversations
    ]
    training_data = [d for d in training_data if d is not None]
    print(f'Tokenized {len(training_data)} examples')

    steps, lr = 50, 1e-4
    for step in range(steps):
        fwd = await training_client.forward_backward_async(training_data, 'cross_entropy')
        opt = await training_client.optim_step_async(tinker.AdamParams(learning_rate=lr))
        result = await fwd.result_async()
        await opt.result_async()
        logprobs = np.concatenate([o['logprobs'].tolist() for o in result.loss_fn_outputs])
        weights_arr = np.concatenate([d.loss_fn_inputs['weights'].tolist() for d in training_data])
        loss = -np.dot(logprobs, weights_arr) / weights_arr.sum()
        print(f'Step {step+1:3d}/{steps}: loss={loss:.4f}')

    save_future = await training_client.save_weights_for_sampler_async('m2-qwen3-4b-sft-v3')
    save_result = await save_future.result_async()
    print(f'New model path: {save_result.path}')

    # Update and re-run step 3 with this new path
    global TINKER_MODEL_PATH
    TINKER_MODEL_PATH = save_result.path
    return save_result.path

new_path = await retrain()
print(f'\nNow re-run cells 3-7 with the new path: {new_path}')